# Memorization DetectionDetect signatures of memorization in model internals. Distinguish memorized facts from genuinely learned patterns, identify which layers store memorized content, and assess extraction risk.

## Why This Matters

Memorization detection identifies when the model is reproducing training data verbatim rather than generalizing. Memorization scores, extractability metrics, and trigger localization are essential for understanding privacy risks and the boundary between memorization and generalization.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jaximport jax.numpy as jnpimport numpy as npfrom irtk import HookedTransformerConfig, HookedTransformercfg = HookedTransformerConfig(n_layers=2, d_model=16, n_ctx=32, d_head=4, n_heads=4, d_vocab=50)model = HookedTransformer(cfg)key = jax.random.PRNGKey(42)leaves, treedef = jax.tree.flatten(model)new_leaves = []for leaf in leaves:    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:        key, sk = jax.random.split(key)        new_leaves.append(jax.random.normal(sk, leaf.shape, dtype=leaf.dtype) * 0.1)    else:        new_leaves.append(leaf)model = jax.tree.unflatten(treedef, new_leaves)tokens = jnp.array([0, 1, 2, 3, 4, 5, 6, 7])def metric(logits):    return float(logits[-1, 0])

In [ ]:
from irtk.memorization_detection import (    memorization_score,    extractability_by_layer,    generalization_gap_profile,    memorized_token_localization,    content_extraction_risk,)

## Memorization ScoreMemorized sequences show high confidence that drops sharply with small input perturbations.

In [ ]:
result = memorization_score(model, tokens)print(f"Clean confidence: {result['clean_confidence']:.4f}")print(f"Confidence drop: {result['confidence_drop']:.4f}")print(f"Memorization score: {result['memorization_score']:.4f}")print(f"Likely memorized: {result['is_likely_memorized']}")

## Extractability by LayerAt which layer does a target token become "accessible" via logit lens?

In [ ]:
result = extractability_by_layer(model, tokens, target_token=5)for l in range(len(result['layer_ranks'])):    print(f"  Layer {l}: rank={result['layer_ranks'][l]}, prob={result['layer_probs'][l]:.6f}")print(f"First extraction layer: {result['extraction_layer']}")print(f"Final rank: {result['final_rank']}")

## Generalization Gap ProfileCompare per-layer predictions on training-like vs test-like inputs.

In [ ]:
train_tokens = jnp.array([0, 1, 2, 3])test_tokens = jnp.array([10, 11, 12, 13])result = generalization_gap_profile(model, train_tokens, test_tokens)for l in range(len(result['train_entropies'])):    print(f"  Layer {l}: train_H={result['train_entropies'][l]:.4f}, "          f"test_H={result['test_entropies'][l]:.4f}, "          f"gap={result['entropy_gap'][l]:.4f}, "          f"rep_dist={result['representation_distance'][l]:.4f}")

## Trigger Position LocalizationWhich token positions, when replaced, cause the largest output change?

In [ ]:
result = memorized_token_localization(model, tokens, metric)for p, imp in enumerate(result['position_importance']):    marker = " <-- TRIGGER" if p in result['trigger_positions'] else ""    print(f"  Pos {p}: importance = {imp:.6f}{marker}")print(f"Most critical: position {result['most_critical_position']}")

## Content Extraction RiskHow confidently does the model continue a sequence? Higher confidence = higher extraction risk.

In [ ]:
prefix = jnp.array([0, 1, 2, 3, 4])target = jnp.array([5, 6, 7])result = content_extraction_risk(model, prefix, target)print(f"Mean probability: {result['mean_probability']:.6f}")print(f"Extraction risk: {result['extraction_risk']}")print(f"Exact match prob: {result['exact_match_prob']:.10f}")print(f"Per-token probs: {[f'{p:.4f}' for p in result['per_token_probs']]}")